# Замеры времени авторских конфигураций

Ноутбук оценивает длительность полного обучения, выполняя короткие запуски с полной
архитектурой, данными и глобальным batch. Фактические эксперименты будут в отдельных
ноутбуках. Здесь обучение на 80 000 шагов не запускается.

На Kaggle выберите GPU T4 ×2 и включите Internet. Выполняйте ячейки сверху вниз;
Save & Run All также подходит. Все внешние процессы явно используют Python 3.11,
поэтому вручную переключать ядро не требуется.

## Создание окружения Python 3.11

Окружение Python 3.11 позволяет установить зафиксированные авторами версии библиотек. Для интерактивной работы можно выбрать зарегистрированное ядро на сервере Kaggle. Для Save & Run All переключать ядро не требуется: все команды установки, токенизации, обучения и замеров ниже вызывают Python этого окружения явно.


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
env_dir = Path("/kaggle/working/mm-env")
env_python = env_dir / "bin" / "python"
if not env_python.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv",
                    "--python", "3.11", "--seed", str(env_dir)], check=True)

subprocess.run([str(env_python), "--version"], check=True)
subprocess.run([str(env_python), "-m", "pip", "install", "ipykernel"], check=True)
subprocess.run([str(env_python), "-m", "ipykernel", "install", "--user",
                "--name", "reproduction-py311",
                "--display-name", "Python 3.11 (reproduction)"], check=True)
print("Создано ядро Python 3.11 (reproduction) в VS Code.")


## Установка кода и библиотек

In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/hw2")
LIBRARY = REPO / "sparse_kernel_regression" / "Library"
PYTHON = Path("/kaggle/working/mm-env/bin/python")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/kostkornilov/hw2.git", str(REPO)], check=True)
assert (LIBRARY / "train_memory_mosaics.py").exists(), "Проверьте путь к Library."
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", "requirements.txt"], cwd=LIBRARY, check=True)
# ninja нужен для сборки авторского CUDA-оператора leaky average.
subprocess.run([str(PYTHON), "-m", "pip", "install", "ninja"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-e", ".", "--no-deps"], cwd=LIBRARY, check=True)


In [28]:
# Выполнить при обновлении кода, перед замером.
subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)


Updating 31dfd29..d2b8908
Fast-forward
 .../Library/memory_mosaics/models/memory_mosaics_eft.py            | 3 ++-
 sparse_kernel_regression/Library/train_memory_mosaics.py           | 7 +++++++
 2 files changed, 9 insertions(+), 1 deletion(-)


From https://github.com/kostkornilov/hw2
   31dfd29..d2b8908  main       -> origin/main


CompletedProcess(args=['git', 'pull', '--ff-only'], returncode=0)

## Language modeling: 1 слой, 3 механизма attention

Измеряем **Gaussian / softmax**, **Epanechnikov / sparsemax** и **1-ReLUmax**,
как выбрано в `reproduction_report.md`. Каждый метод использует обе T4;
методы запускаются последовательно.

#### Скачивание и распаковка BabiStories

Готовый датасет находится в оригинальном репозитории [Memory Mosaics](https://github.com/facebookresearch/MemoryMosaics/tree/main/BabiStories/data). Данные сохраняются в `/kaggle/working/babistories`.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("/kaggle/working/babistories")
data_dir.mkdir(parents=True, exist_ok=True)
base_url = (
    "https://raw.githubusercontent.com/facebookresearch/"
    "MemoryMosaics/main/BabiStories/data"
)

for i in range(1, 6):
    name = f"babistories-dataset.7z.{i:03d}"
    if (data_dir / name).exists():
        continue
    print("Downloading", name, flush=True)
    # Сначала скачиваем во временный файл, чтобы не принять неполную загрузку за архив.
    temporary = data_dir / (name + ".part")
    urlretrieve(f"{base_url}/{name}", temporary)
    temporary.replace(data_dir / name)


In [ ]:
import shutil
import subprocess

# Устанавливаем архиватор только если он отсутствует в сессии.
if shutil.which("7z") is None:
    subprocess.run(["sudo", "apt-get", "update"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", "p7zip-full"], check=True)

# 7z сам прочитает остальные части архива из той же папки.
subprocess.run([
    "7z", "x", str(data_dir / "babistories-dataset.7z.001"),
    f"-o{data_dir}", "-aos",
], check=True)


In [24]:
for path in sorted(data_dir.rglob("*.txt")):
    print(path.relative_to(data_dir), f"{path.stat().st_size / 1024**2:.1f} MiB")


traindataset.txt 1880.0 MiB
valdataset.txt 18.8 MiB


#### Токенизация готовых train/val

В распакованном архиве уже есть `traindataset.txt` и `valdataset.txt`; разбиение не меняем. Запускаем потоковую версию авторского `prepare.py`: GPT-2 tokenizer, EOT после каждой истории, массивы uint16. Файл `dataset_stats.json` содержит количество историй, токенов и SHA256 исходных текстов. Таблица 4 статьи сообщает 474 704 907 train-токенов и 4 749 107 validation-токенов; расхождение нужно проверить до замеров.


In [ ]:
import json
import shutil

TOKEN_DIR = LIBRARY / "memory_mosaics" / "data" / "BabiStories"
STATS_PATH = TOKEN_DIR / "dataset_stats.json"
if not (STATS_PATH.exists() and (TOKEN_DIR / "train.bin").exists() and (TOKEN_DIR / "val.bin").exists()):
    subprocess.run([
        str(PYTHON), str(TOKEN_DIR / "prepare.py"),
        "--train-file", str(data_dir / "traindataset.txt"),
        "--val-file", str(data_dir / "valdataset.txt"),
        "--output-dir", str(TOKEN_DIR),
    ], check=True)
stats = json.loads(STATS_PATH.read_text())
print(json.dumps(stats, indent=2))
for split, expected in {"train": 474704907, "val": 4749107}.items():
    actual = stats[split]["tokens"]
    assert actual == expected, f"{split}: {actual} токенов вместо {expected}; проверьте данные."


### Полная конфигурация и длина замера

Параметры полной конфигурации приведены в [статье, приложение B.1, таблица 3](https://arxiv.org/html/2601.22766v3#A2.SS1).

| Параметр | Значение |
|---|---|
| Слои / embedding / головы | 1 / 768 / 12 |
| Persistent memory / контекст | 2688 / 512 |
| Глобальный batch | 512 (microbatch 8 × 2 GPU × accumulation 32) |
| Шаги обучения | 80 000 |
| Learning rate | 5e-3 → 1e-4; warmup 2 000 шагов |
| Расписание learning rate | Cosine decay до шага 80 000 |
| Weight decay | 0.1 |
| Adam β | (0.9, 0.95) |
| Ограничение нормы градиента | 1 |
| Численная точность | FP16 + GradScaler на T4; в статье — BF16 на H100/H200 |
| Attention backend | Sparsemax — AdaSplash 0.1.1b0 / Triton 3.0.0; softmax — PyTorch SDPA; 1-ReLUmax — операции авторов на PyTorch |
| Оптимизации | `torch.compile`, авторский CUDA leaky-average и fused AdamW |
| Короткий замер | 24 шага; первые 8 исключаются из оценки скорости |
| Validation и сохранение checkpoint | Дважды за короткий запуск; `eval_iters=10` |
| Расписание LR в коротком запуске | Сохраняется без сжатия |

Короткий замер оценивает скорость и накладные расходы, но не итоговое качество модели.


Sparsemax использует AdaSplash 0.1.1b0 / Triton 3.0.0; softmax — PyTorch SDPA;
1-ReLUmax — авторские операции PyTorch. Для всех включены `torch.compile`,
авторский CUDA leaky-average и fused AdamW. На T4 используем FP16 + GradScaler
вместо авторского BF16 на H100/H200. Microbatch 8 × 2 GPU × accumulation 32 = 512.

Короткий запуск выполняет 24 полных шага, первые 8 исключаются из оценки скорости.
Validation с авторским `eval_iters=10` и сохранение checkpoint выполняются дважды,
чтобы измерить их стоимость. Расписание LR сохраняется авторским, без сжатия.
Замер скорости не оценивает итоговое качество модели.

In [33]:
from datetime import datetime

METHODS = ["softmax", "epanechnikov", "relumax"]
LABELS = {"softmax": "Gaussian / softmax", "epanechnikov": "Epanechnikov / sparsemax",
          "relumax": "1-ReLUmax"}
GPU_IDS = [0, 1]
MICRO_BATCH = 8
GLOBAL_BATCH = 512
MEASURE_STEPS = 24
WARMUP_STEPS = 8  # Прогрев для замера, не LR warmup модели.
FULL_STEPS = 80000
FULL_EVAL_INTERVAL = 1000

assert GLOBAL_BATCH % (MICRO_BATCH * len(GPU_IDS)) == 0
assert MEASURE_STEPS % 2 == 0 and 0 < WARMUP_STEPS < MEASURE_STEPS // 2
ACCUMULATION = GLOBAL_BATCH // (MICRO_BATCH * len(GPU_IDS))
RESULTS = Path("/kaggle/working/results/timing/language_modeling") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RESULTS.mkdir(parents=True, exist_ok=True)

# Параметры полного обучения. В коротком запуске ниже меняются только
# число шагов, частота evaluation и частота печати лога.
FULL_CONFIG = {
    "seed": 1337, "attention_backend": "adasplash", "dtype": "float16", "compile": True,
    "n_layer": 1, "n_embd": 768, "n_head": 12, "pmem_size": 2688, "block_size": 512,
    "batch_size": MICRO_BATCH, "gradient_accumulation_steps": ACCUMULATION,
    "max_iters": FULL_STEPS, "learning_rate": 5e-3, "min_lr": 1e-4,
    "weight_decay": 0.1, "beta1": 0.9, "beta2": 0.95, "grad_clip": 1.0,
    "warmup_iters": 2000, "lr_decay_iters": 80000, "decay_type": "cosine",
    "eval_interval": FULL_EVAL_INTERVAL, "eval_iters": 10, "log_interval": 100,
    "save_checkpoint_interval": FULL_STEPS,
}
print(f"1 слой; global batch: {MICRO_BATCH} × {len(GPU_IDS)} × {ACCUMULATION} = {GLOBAL_BATCH}")
print(f"Замер: {MEASURE_STEPS} шагов на метод; прогноз: {FULL_STEPS:,} шагов на метод.")
print("Новый каталог замеров:", RESULTS)

1 слой; global batch: 8 × 2 × 32 = 512
Замер: 24 шагов на метод; прогноз: 80,000 шагов на метод.
Новый каталог замеров: /kaggle/working/results/timing/language_modeling/20261010_063135_385657


### Проверка окружения

Версии зафиксированы в авторском requirements. При ошибке AdaSplash или компиляции
замер останавливается: автоматического перехода на другой backend нет.

In [34]:
import json
import shutil

subprocess.run([str(PYTHON), "-c", f"""
import torch
from importlib.metadata import version
from adasplash import adasplash_no_block_mask
assert torch.cuda.is_available()
assert version("torch").split("+")[0] == "2.4.1"
assert version("adasplash") == "0.1.1b0"
assert version("triton") == "3.0.0"
for i in {GPU_IDS!r}:
    assert 0 <= i < torch.cuda.device_count()
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("torch:", torch.__version__, "AdaSplash:", version("adasplash"), "Triton:", version("triton"))
"""], check=True)
(RESULTS / "environment.txt").write_text(
    subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True))
shutil.copy2(STATS_PATH, RESULTS / "dataset_stats.json")

0 Tesla T4 (7, 5)
1 Tesla T4 (7, 5)
torch: 2.4.1+cu121 AdaSplash: 0.1.1b0 Triton: 3.0.0


PosixPath('/kaggle/working/results/timing/language_modeling/20261010_063135_385657/dataset_stats.json')

### Короткие запуски и сбор замеров

Первые шаги могут занимать существенно больше времени из-за компиляции.
`remain time` в логе относится только к короткому запуску. Оценка полного обучения
появится в следующей итоговой таблице, отдельно для каждого метода.

Каждый повтор ячейки конфигурации создаёт новый каталог. В существующий checkpoint
замер не продолжается. Если возникает OOM, уменьшите microbatch в конфигурации:
accumulation пересчитается с сохранением batch 512.

In [ ]:
import math
import os
import re
import statistics
import time


def benchmark(method):
    out_dir = RESULTS / "runs"
    run_dir = out_dir / method
    assert not run_dir.exists(), "Для повторного замера выполните ячейку конфигурации заново."
    run_dir.mkdir(parents=True)
    parameters = {**FULL_CONFIG, "method": method, "out_dir": str(out_dir),
                  "max_iters": MEASURE_STEPS, "eval_interval": MEASURE_STEPS // 2,
                  "log_interval": 1}
    command = [str(PYTHON), "-m", "torch.distributed.run", "--standalone",
               f"--nproc_per_node={len(GPU_IDS)}", "train_memory_mosaics.py"]
    for key, value in parameters.items():
        command += [f"--{key}", str(value)]
    (run_dir / "run_config.json").write_text(json.dumps({
        "full_config": FULL_CONFIG, "benchmark_parameters": parameters,
        "command": command, "gpu_ids": GPU_IDS,
        "repository_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    }, indent=2))
    environment = os.environ.copy()
    environment.update(CUDA_VISIBLE_DEVICES=",".join(map(str, GPU_IDS)),
                       PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")
    environment.setdefault("MAX_JOBS", "2")
    print(f"\n{LABELS[method]}: {MEASURE_STEPS} шагов на {len(GPU_IDS)} GPU", flush=True)
    started = time.monotonic()
    with subprocess.Popen(command, cwd=LIBRARY, env=environment,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    elapsed = time.monotonic() - started

    log = (run_dir / "train.log").read_text()
    samples = [(int(step), float(ms) / 1000) for step, ms in re.findall(
        r"iter (\d+): loss .*?time ([\d.]+)ms / step", log)]
    assert [step for step, _ in samples] == list(range(MEASURE_STEPS))
    stable = [seconds for step, seconds in samples if step >= WARMUP_STEPS]
    evaluation = [float(value) for value in re.findall(
        r"\[timing\] evaluation \+ checkpoint: ([\d.]+)s", log)]
    assert len(evaluation) == 2, "Нужен обновлённый train_memory_mosaics.py с замерами evaluation."
    history = json.loads((run_dir / "loss_history.json").read_text())
    assert history["iter"][-1] == MEASURE_STEPS and (run_dir / "ckpt.pt").exists()
    assert all(math.isfinite(value) for field in ("train", "val") for value in history[field])
    step_seconds = statistics.median(stable)
    eval_seconds = statistics.mean(evaluation)
    # Остаток wall time учитывает запуск процессов, компиляцию и прогрев.
    startup_seconds = max(0, elapsed - MEASURE_STEPS * step_seconds - sum(evaluation))
    training_seconds = FULL_STEPS * step_seconds
    evaluation_count = (FULL_STEPS - 1) // FULL_EVAL_INTERVAL + 1
    result = {
        "method": method, "step_seconds": step_seconds,
        "step_min_seconds": min(stable), "step_max_seconds": max(stable),
        "evaluation_checkpoint_seconds": eval_seconds,
        "startup_warmup_seconds": startup_seconds, "benchmark_wall_seconds": elapsed,
        "full_steps": FULL_STEPS, "evaluation_count": evaluation_count,
        "training_hours": training_seconds / 3600,
        "full_hours": (training_seconds + evaluation_count * eval_seconds + startup_seconds) / 3600,
    }
    (run_dir / "timing.json").write_text(json.dumps(result, indent=2))
    return result


timings = []
for method in METHODS:
    timings.append(benchmark(method))


Gaussian / softmax: 24 шагов на 2 GPU


### Сколько длилось бы полное обучение

«Только обучение» = медиана времени шага после прогрева × 80 000.
«С evaluation и сохранением» дополнительно учитывает 80 оценок train/val
с checkpoint и одноразовые затраты запуска/прогрева.

Это экстраполяция короткого замера, а не измеренные часы полного обучения.
Разброс времени шага показан в таблице. Стоимость компиляции зависит от кеша;
оценка не включает установку окружения, загрузку и токенизацию данных,
перезапуски сессий или загрузку checkpoints. Стоимость сохранений со временем
может меняться. Строка «Все три» относится к последовательным запускам
каждого метода на обеих GPU.

In [ ]:
import csv
from IPython.display import Markdown, display

# Читаем сохранённые замеры: ячейка работает и после перезапуска ядра,
# если RESULTS указывает на каталог завершённого замера.
timings = [json.loads((RESULTS / "runs" / method / "timing.json").read_text())
           for method in METHODS]
with (RESULTS / "timing_summary.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(timings[0]))
    writer.writeheader()
    writer.writerows(timings)

rows = ["| Метод | Секунд/шаг | Диапазон после прогрева, с | Только обучение, ч | С evaluation и сохранением, ч |",
        "|---|---:|---:|---:|---:|"]
for result in timings:
    rows.append(f"| {LABELS[result['method']]} | {result['step_seconds']:.3f} | "
                f"{result['step_min_seconds']:.3f}–{result['step_max_seconds']:.3f} | "
                f"{result['training_hours']:.1f} | {result['full_hours']:.1f} |")
rows.append(f"| **Все три последовательно** | — | — | "
            f"**{sum(r['training_hours'] for r in timings):.1f}** | "
            f"**{sum(r['full_hours'] for r in timings):.1f}** |")
display(Markdown("\n".join(rows)))
print("Таблица и исходные замеры:", RESULTS)

После Save & Run All артефакты доступны в outputs версии ноутбука:
`results/timing/language_modeling/<дата_замера>/`. Для отчёта достаточно
`timing_summary.csv`, файлов `timing.json`, `run_config.json`, `train.log`,
`environment.txt` и `dataset_stats.json`. Checkpoints коротких запусков
сохраняются обучающим скриптом, но не являются результатами полного воспроизведения.

### Проверка артефактов на коротких запусках

Выполните следующие две ячейки после завершения замеров всех трёх методов.
Повторное обучение не требуется. После перезапуска ядра можно явно задать
`ARTIFACT_ROOT` путём к каталогу замера; по умолчанию используется `RESULTS`,
а если он не задан — самый новый каталог замеров.

Проверяем checkpoint (веса, optimizer, GradScaler, конфигурацию и последний шаг),
согласованность JSON и CSV и конечность loss и весов. Создаём итоговую таблицу
`results_summary.csv` / `.md`, графики `loss_curves.png` / `.pdf` и
`artifact_validation.json` с размерами и SHA256 исходных артефактов.
Проверка чтения checkpoint ещё не проверяет продолжение обучения из него.

Таблица и графики короткого замера помечаются `timing-smoke`: это проверка
сохранения результатов, а не сравнение итогового качества со статьёй.


In [38]:
from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/hw2")
PYTHON = Path("/kaggle/working/mm-env/bin/python")
METHODS = ["softmax", "epanechnikov", "relumax"]
# При необходимости замените None на Path("/kaggle/working/results/timing/language_modeling/...").
ARTIFACT_ROOT = None
if ARTIFACT_ROOT is None:
    if "RESULTS" in globals():
        ARTIFACT_ROOT = Path(RESULTS)
    else:
        candidates = sorted(Path("/kaggle/working/results/timing/language_modeling").glob("*/runs"))
        assert candidates, "Не найден каталог замеров; задайте ARTIFACT_ROOT явно."
        ARTIFACT_ROOT = candidates[-1].parent
print("Проверяем:", ARTIFACT_ROOT)
EXPORT_SCRIPT = REPO / "scripts" / "export_lm_artifacts.py"
assert EXPORT_SCRIPT.exists(), "Обновите репозиторий в Kaggle через git pull --ff-only."
subprocess.run([str(PYTHON), str(EXPORT_SCRIPT), str(ARTIFACT_ROOT),
                "--kind", "timing-smoke", "--methods", *METHODS], check=True)


Проверяем: /kaggle/working/results/timing/language_modeling/20261010_063135_385657


AssertionError: Обновите репозиторий в Kaggle через git pull --ff-only.

In [ ]:
import json
from IPython.display import Image, Markdown, display

display(Markdown((ARTIFACT_ROOT / "results_summary.md").read_text()))
display(Image(filename=str(ARTIFACT_ROOT / "loss_curves.png")))
validation = json.loads((ARTIFACT_ROOT / "artifact_validation.json").read_text())
assert validation["status"] == "passed"
print("Проверка пройдена. Исходные артефакты:")
for name, item in validation["artifacts"].items():
    print(f"{name}: {item['bytes'] / 1024**2:.2f} MiB")
print("Все файлы для скачивания:", ARTIFACT_ROOT)


### Скачивание артефактов

Ячейка создаёт ZIP с таблицами, графиками, логами, конфигурациями и результатом
проверки. Checkpoints `.pt` не входят в ZIP из-за размера; ссылки на них показаны
отдельно. Нажмите ссылку на ZIP в выводе ячейки.

Если ссылки недоступны в интерфейсе подключённого ядра VS Code, откройте ноутбук
в Kaggle: файлы лежат в `/kaggle/working`. После Save Version → Save & Run All
их можно скачать из раздела Output сохранённой версии.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

assert (ARTIFACT_ROOT / "artifact_validation.json").exists(), "Сначала выполните проверку артефактов."
archive = Path("/kaggle/working") / f"lm_artifacts_{ARTIFACT_ROOT.name}.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file() and path.suffix != ".pt":
            bundle.write(path, arcname=str(Path(ARTIFACT_ROOT.name) / path.relative_to(ARTIFACT_ROOT)))
print(f"Архив для отчёта: {archive} ({archive.stat().st_size / 1024**2:.2f} MiB)")
display(FileLink(str(archive)))
print("Checkpoints отдельно:")
for method in METHODS:
    checkpoint = ARTIFACT_ROOT / "runs" / method / "ckpt.pt"
    print(f"{method}: {checkpoint.stat().st_size / 1024**2:.1f} MiB")
    display(FileLink(str(checkpoint)))


## In-context learning

Замер авторских конфигураций RegBench будет добавлен следующим этапом.

## Length generalization

Замер авторских синтетических конфигураций будет добавлен следующим этапом.